# Labor Risk Profiling: Predicting Income Inequality

## 1. Introduction & Data Loading
The goal of this project is to build a machine learning classification model capable of predicting whether an individual's income exceeds $50K/year based on census data. We will utilize first a Random Forest Classifier and deeply analyze the model's decision-making process using advanced interpretability techniques (SHAP, PDP, Permutation Importance). Then we'll compare it with Xgboost, clustering and deep learning algorithm.

First, we import the necessary libraries and load the dataset.
The dataset has been taken from
https://www.kaggle.com/datasets/sagnikpatra/uci-adult-census-data-dataset



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
input_file = 'labor_risk_profiling_dataset\\adult.csv'
df=pd.read_csv(input_file)


## 2. Exploratory Data Analysis (EDA)
Before training the model, it is crucial to understand the underlying distribution of our data. In this section, we will:
* Check for missing values and handle them.
* Analyze the distribution of the target variable (`>50K` vs `<=50K`) to check for class imbalance.
* Visualize the relationships between key numerical/categorical features and the target variable.

In [ ]:
df_1=df.copy()

In [ ]:
df_1=df_1.replace('?', np.nan, inplace=True)

In [ ]:
# df_1.isnull().sum()

In [ ]:
# df_1.shape

### 2.1 Missing analysis with matrix

In [ ]:
import missingno as msno
msno.matrix(df_1)

As shown in the missing data matrix, there is a perfect overlap in the missing values for workclass and occupation. This structural missingness suggests that when an individual's employment status is unknown or unrecorded, both their work sector and specific profession are naturally omitted.

### 2.2 Bivariate Analysis: Exploring Working Hours

In this series of visualizations, we explore how various demographic and educational features correlate with the number of hours worked per week (`hours.per.week`).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig,axes=plt.subplots(2,2,figsize=(15,10), sharey=True)

#primo grafico
sns.boxplot(x='education.num', y='hours.per.week',data=df_1, ax=axes[0,0])
axes[0,0].set_title('Boxplot di Hours per Week per Education')
axes[0,0].set_xlabel('Education')
axes[0,0].set_ylabel('Hours per Week')
axes[0,0].tick_params(axis='x', rotation=45)

#secondo grafico
sns.violinplot(x='sex', y='hours.per.week',data=df_1, ax=axes[0,1])
axes[0,1].set_title('Violin plot of sex and Hours per Week')
axes[0,1].set_xlabel('sex')
axes[0,1].set_ylabel('Hours per Week')
axes[0,1].tick_params(axis='x', rotation=45)

#terzo grafico
sns.scatterplot(x='age', y='hours.per.week',data=df_1, ax=axes[1,0])
axes[1,0].set_title('Scatterplot of Age and Hours per Week')
axes[1,0].set_xlabel('Age')
axes[1,0].set_ylabel('Hours per Week')

#quarto grafico
sns.stripplot(x='education.num', y='hours.per.week',data=df_1, jitter=True, ax=axes[1,1])
axes[1,1].set_title('Strip plot of Education and Hours per Week')
axes[1,1].set_xlabel('Education')
axes[1,1].set_ylabel('Hours per Week')

plt.tight_layout()

### 2.3 Income Probability by Age and Gender

In [ ]:
# definizione delle fasce d'età
bins = [0, 25, 35, 45, 55, 65, 100]
labels = ['0-25', '26-35', '36-45', '46-55', '56-65', '66+']
df_1['age_bins'] = pd.cut(df_1['age'], bins=bins, labels=labels)
# Trasforma la colonna in categoria ordinata
df_1['age_bins'] = pd.Categorical(df_1['age_bins'], categories=labels, ordered=True)

In [ ]:
df_1['is_high_income']=df_1['income'].apply( lambda x: 0 if x=='<=50K' else 1)

In [ ]:
# mean number of people with income > 50K for sex and age_bins
pivot_income = df_1.pivot_table(values='is_high_income', index='age_bins', columns='sex', aggfunc='mean')
sns.heatmap(pivot_income.iloc[::-1], annot=True, cmap='YlGnBu')

This heatmap highlights the probability of an individual earning above the $50K income threshold, grouped by age bins and sex.

The most evident takeaway from this visualization is the stark **gender inequality** regarding high-income possibilities. Across every single age bracket, men have a significantly higher probability of reaching the >$50K threshold compared to women.

* **The Peak Earning Years:** The income gap becomes especially pronounced during the core career years (ages 40 to 60). In these age bins, the probability of high income for men peaks at **43-45%**, whereas for women in the exact same age groups, it never exceeds **18%**.

### 2.4 High-Income Likelihood by Education Level and Sex

In [ ]:
# pivot_education_income=pd.pivot_table(values='is_high_income', index='education.num', columns='sex', aggfunc='mean', data=df_1)

In [ ]:
# sns.heatmap(pivot_education_income,annot=True)

In [ ]:
sns.set_theme(style="whitegrid")
g = sns.catplot(
    data=df_1,
    kind="bar",
    x="education.num",
    y="is_high_income",
    hue="sex",
    col="age_bins",
    errorbar=None,
    col_wrap=3,          # Numero di grafici per riga
    palette="viridis",    # Schema colori
    height=4,
    aspect=1.2
)

This faceted catplot illustrates the probability of high income across various age cohorts, segmented by gender and education level. The data demonstrates a persistent gender disparity that evolves over time; while the 'income gap' is present in younger years, it becomes drastically more pronounced during peak earning years (ages 36–65). Even at the highest levels of education, women consistently face a lower probability of high-income attainment than their male counterparts in every age bin

### 2.5 Exploratory Plot: Assessing Feature Distribution and Correlation

In [ ]:
# individuo le variabili numeriche e categoriche
def identify_variable_types(df):
    numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
    return numeric_features, categorical_features

In [ ]:
result=identify_variable_types(df_1)

In [ ]:
import math

In [ ]:
def plot_outliers(df, lista):
        # Calcoliamo quante righe servono per avere 3 colonne al massimo
    n_cols = 3
    n_rows = math.ceil(len(lista) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten() # Lo rendiamo 1D per gestirlo facilmente

    for i, col in enumerate(lista):
        sns.boxplot(data=df, x=col, ax=axes[i], color='skyblue', fliersize=5)
        axes[i].set_title(f'Focus Outlier: {col}', fontsize=14)
        axes[i].grid(True, linestyle='--', alpha=0.6)

    # Nascondiamo i grafici vuoti se la lista non riempie l'ultima riga
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_outliers(df_1, result[0])

In [ ]:
df_num=df_1.drop(columns=result[1])
ax=df_num.hist(figsize=(15,10), bins=20, color='skyblue', edgecolor='black')

In [ ]:
df_1_sample=df_1[df_num.columns].sample(1000, random_state=42)

In [ ]:
sns.pairplot(df_1_sample, vars=df_num.columns, hue='is_high_income', palette='viridis')

The presence of significant skewness and numerous outliers in variables like capital.gain and fnlwgt suggests that data scaling or normalization may be necessary before training a machine learning model. Additionally, the class imbalance in the target variable (is_high_income) should be addressed to avoid model bias.

### 2.6 Spearman Correlation Matrix

In [ ]:
corr_spearman=df_1[result[0]].corr(method='spearman')
sns.heatmap(corr_spearman, annot=True, cmap='coolwarm', center=0)
plt.title('Spearman Correlation Matrix')
plt.show()

The Spearman Correlation Matrix identifies education.num and capital.gain as the strongest predictors of the high-income target variable. While the coefficients indicate moderate positive correlations, they are significantly higher than other features. Additionally, age and hours.per.week show secondary levels of correlation, suggesting they are also relevant contributors to the model's predictive power.

### 2.7 Contingency table

In [ ]:
# Creiamo la tabella di contingenza grezza
contingency_table = pd.crosstab(df_1['race'], df_1['is_high_income'])

print("Contingency table (Absolute Frequencies):")
print(contingency_table)

The contingency table illustrates the distribution of income levels across different ethnic groups. While the absolute number of high earners is highest within the 'White' group, a proportional analysis is required to account for the significant class imbalance. Preliminary observations suggest varying rates of high-income attainment across groups, which may indicate systemic economic disparities that warrant further statistical testing, such as a Chi-Square test.

In [ ]:
#trasformo in frequenze relative
# Normalizziamo per riga (ogni riga somma a 1.0, ovvero 100%)
ct_normalized = pd.crosstab(df_1['race'], df_1['is_high_income'], normalize='index')

# Creiamo lo Stacked Bar Chart
ct_normalized.plot(kind='bar', stacked=True, figsize=(12, 6), color=['#ff9999', '#66b3ff'])
plt.title("Relative Distribution of Income Categories by Ethnic Group")
plt.ylabel("Percentages (0.0 - 1.0)")
plt.xlabel("Ethnic Group")
plt.legend(title="Income >50k", labels=["No", "Yes"])
plt.xticks(rotation=45)
plt.show()

The normalized stacked bar chart illustrates the proportional distribution of income categories within each ethnic group. By standardizing the data, we can observe that the 'Asian-Pac-Islander' and 'White' groups exhibit the highest relative frequencies of high-income attainment (blue). In contrast, the 'Other', 'Black', and 'Amer-Indian-Eskimo' groups show a significantly higher proportion of individuals in the lower-income bracket (red), highlighting a clear economic disparity that is independent of total group size.

In [ ]:
# check statiscal significance with chi2 test
from scipy.stats import chi2_contingency

# Eseguiamo il test sulla tabella dei FATTI (quella non normalizzata dello Step 1)
chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"Statistic Chi-Square: {chi2:.2f}")
print(f"P-Value: {p:.4e}") # Notazione scientifica perché sarà molto piccolo

In [ ]:
# cramer's V
n = contingency_table.sum().sum() # Totale campioni
min_dim = min(contingency_table.shape) - 1 # Minimo tra (righe-1) e (colonne-1)
v_cramer = np.sqrt(chi2 / (n * min_dim))

print(f"Cramer's V: {v_cramer:.4f}")

Statistical tests confirm the existence of an economic disparity across ethnic groups. The extremely low p-value indicates that the relationship is statistically significant, while the Cramer's V (0.10) suggests that, although ethnicity influences income, the strength of this association is modest. This indicates that other socioeconomic factors contribute simultaneously to determining income levels.

In [ ]:


def analyze_all_categorical(df, cat_columns, target_col):
    results = []

    for col in cat_columns:
        # 1. Tabella di contingenza
        ct = pd.crosstab(df[col], df[target_col])

        # 2. Test Chi-Quadro
        chi2, p, dof, expected = chi2_contingency(ct)

        # 3. Calcolo V di Cramer
        n = ct.sum().sum()
        min_dim = min(ct.shape) - 1
        v_cramer = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0

        # Salviamo i risultati in un dizionario
        results.append({
            'Variable': col,
            'Chi-Square': round(chi2, 2),
            'P-Value': f"{p:.4e}",
            'Cramer\'s V': round(v_cramer, 4),
            'Significant': "Yes" if p < 0.05 else "No"
        })

    # Creiamo il DataFrame finale e ordiniamo per V di Cramer (più alta = più importante)
    summary_df = pd.DataFrame(results).sort_values(by='Cramer\'s V', ascending=False)
    return summary_df



In [ ]:
# Esecuzione
tabella_risultati_cat = analyze_all_categorical(df_1, result[1], 'is_high_income')
tabella_risultati_cat

The summary table of Chi-Square and Cramer's V tests reveals that all analyzed variables are statistically significant (p-value $\approx$ 0). However, the strength of these associations varies considerably. Relationship and Marital Status exhibit the strongest associations with income (Cramer's V $\approx$ 0.45), followed closely by Education and Occupation. Conversely, variables such as Race and Native Country, while statistically significant, show the weakest practical association (Cramer's V $\approx$ 0.10). This suggests that while demographic factors play a role, household structure and human capital (education/job) are far more powerful predictors of high-income attainment.

In [ ]:
# Creiamo la tabella di contingenza grezza
con_marital_age = pd.crosstab(df_1['marital.status'], df_1['age_bins'])
print(f"Tabella di contingenza:\n {con_marital_age}")

In [ ]:
#trasformo in frequenze relative
# Normalizziamo per riga (ogni riga somma a 1.0, ovvero 100%)
ct_norm_marital_age = pd.crosstab(df_1['marital.status'], df_1['age_bins'], normalize='index')
ct_norm_marital_age

In [ ]:
ct_norm_marital_age.plot(kind='bar',stacked=True, figsize=(12, 6), color=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6'])

In [ ]:

chi2, p, dof, expected = chi2_contingency(con_marital_age)
print(f"Statistica Chi-Quadro: {chi2:.2f}")
print(f"P-Value: {p:.4e}")
print(f"grado di libertà: {dof}")

In [ ]:
# cramer's V
n = con_marital_age.sum().sum() # Totale campioni
n

In [ ]:
min_dim = min(con_marital_age.shape) - 1 # Minimo tra (righe-1) e (colonne-1)
v_cramer = np.sqrt(chi2 / (n * min_dim))

print(f"V di Cramer: {v_cramer:.4f}")

Il collegamento con la tua EDA
Il ColumnTransformer è dove metti in pratica i tuoi risultati:

Hai visto V di Cramer 0.30 tra marital-status e age? Ecco perché nel codice sopra le ho tenute entrambe.

Hai visto che capital-gain è schiacciato nei boxplot? Usi RobustScaler invece di StandardScaler.

Hai visto che income ha V di Cramer 0.99? Lo lasci fuori dal trasformatore (o usi remainder='drop').

## 3. Data Preprocessing & Pipeline Construction
Machine learning models require numerical input. Here, we build a Scikit-Learn `Pipeline` to handle data transformations automatically and prevent data leakage:
* **Numerical Features:** Imputing missing values and scaling.
* **Categorical Features:** Applying One-Hot Encoding to transform text labels into a machine-readable format.

In [ ]:
# Identifichiamo i paesi rari (meno di 100 occorrenze)
counts = df_1['native.country'].value_counts()
rare_countries = counts[counts < 100].index


In [ ]:
# Li sostituiamo con 'Other'
df_1['native.country'] = df_1['native.country'].replace(rare_countries, 'Other')

### 3.1 Data Preparation: Feature and Target Separation
Separate the dataset into the Features ($X$) and the Target ($y$) to prepare for supervised learning

In [ ]:
X=df_1.drop(columns=['income', 'is_high_income'])
y=df_1['is_high_income']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Dividiamo subito in Train e Test (fondamentale per evitare Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 3.2 ColumnTransformer Definition

Using ColumnTransformer to apply specific preprocessing pipelines to each feature type:

- Numerical Features: Applied RobustScaler to handle the high volume of outliers detected during EDA.
- Categorical Features: Applied OneHotEncoder to convert qualitative categories into a machine-readable format.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline

In [ ]:
#identificazione colonne numeriche e categoriche
num_features=[x for x in result[0] if x not in ['is_high_income', 'fnlwgt']]


In [ ]:
cat_features=[x for x in result[1] if x not in ['income','age_bins']]


### 3.3 Preprocessor

In [ ]:
# Creiamo il preprocessore
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
    ]
)

### 3.4 Full Pipeline (Preprocessing + Model)
We consolidate the preprocessor and a Random Forest Classifier into a single Pipeline object.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Creiamo la pipeline finale
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

# Alleniamo tutto il blocco in una riga sola!
model_pipeline.fit(X_train, y_train)

### 3.5 Random forest results

In [ ]:
from sklearn.metrics import classification_report

y_pred = model_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

#### 3.5.1 Model Performance Analysis
The results are solid for an initial run on the Adult dataset. An 84% accuracy is standard, but the real insights lie in the class-specific metrics:
- **Majority Class** ($\le$50k): High performance (88% Precision / 91% Recall). The model is excellent at identifying low-to-mid earners because they represent the majority of the data.
- **Minority Class** (>50k): Performance drops (0.68 Precision / 0.63 Recall). The model struggles to isolate the specific signals of high earners, missing about 37% of them (Recall) and occasionally misidentifying low earners as "rich" (Precision).
- **Generalization**: The 84% accuracy suggests the model is generalising well. By grouping native-country during preprocessing, you prevented overfitting, ensuring the model learned meaningful patterns rather than memorizing noise from rare categories.

**Conclusion**: The model is honest and stable, though it remains slightly biased toward the majority class—a common challenge in imbalanced datasets.

#### 3.5.2 Hyperparameter Tuning: Grid Search
We use Grid Search to find the optimal configuration for the Random Forest:

- Objective: Systematically test combinations of parameters (like n_estimators and max_depth).

- Method: Employs Cross-Validation to ensure the best parameters are consistent across different data folds.

- Goal: Maximize predictive performance and reduce bias toward the majority class.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
#definiamo i parametri da testare
param_grid = {
    'classifier__n_estimators': [100],
    'classifier__max_depth': [None, 10],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2],
    'classifier__class_weight': ['balanced']
}


We tune the following parameters to optimize the bias-variance tradeoff and handle class imbalance:

- n_estimators [100, 200]: Number of trees. More trees increase stability but also computation time.

- max_depth [None, 10, 20]: Maximum tree depth. Limiting depth prevents trees from becoming overly complex and overfitting.

- min_samples_split [2, 5]: Minimum samples required to split a node. Higher values lead to more conservative trees.

- min_samples_leaf [1, 2]: Minimum samples per leaf. Increasing this value smooths the model, especially in noisy data. Higher values ensure every conclusion the tree draws is backed by enough evidence, making it less sensitive to noise.

- class_weight ['balanced', 'balanced_subsample']: Adjusts weights inversely proportional to class frequencies to give more importance to the minority class (high income).

In [ ]:
# Definiamo una griglia di iperparametri da testare
from sklearn.model_selection import RandomizedSearchCV


random_search = RandomizedSearchCV(
    model_pipeline,
    param_grid,
    n_iter=12,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# ho ridotto a cv 3
# scelto randomized al posto di grid Invece di provare tutte le 48 combinazioni, ne campiona un numero fisso a caso
# tolto dai parametri alcuni settaggi per ridurre i tempi

In [ ]:

random_search.fit(X_train, y_train)

In [ ]:
# best_model e best_params sono attributi che grid e random popolano automaticamente

print(f"Best parameters found: {random_search.best_params_}")
best_model = random_search.best_estimator_

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predizione con il miglior modello
y_pred = best_model.predict(X_test)

# Generazione della matrice
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['<=50K', '>50K'])

disp.plot(cmap='Blues')
plt.title("Confusion Matrix of the Best Random Forest")
plt.show()

- True Negatives (4206): The model correctly identified almost everyone earning ≤$50K. It is very reliable on the majority class.
- True Positives (1242): The model correctly flagged most high-income individuals — a strong result given how underrepresented this class is in the data.
- False Negatives (326): These are the "missed rich" — people the model predicted as low-income when they actually earn >$50K. This number being low means the model has strong Recall: it rarely lets a true positive slip through.
- False Positives (739): These are the "false alarms" — people predicted as high-income who actually earn ≤$50K. This is the main weakness of the model.

Technical note: Using class_weight='balanced' makes the model more aggressive in hunting for high-income cases. This is a deliberate tradeoff: the model accepts more False Positives in exchange for fewer False Negatives — in other words, it would rather over-predict rich than miss them entirely. Whether this tradeoff is acceptable depends on the cost of each type of error in your specific use case.

#### 3.5.3 Global interpretation

##### 3.5.3.1 Permutation Importance
Allows you to immediately display a ranking of the most important features for your Random Forest across the entire dataset.

In [ ]:

from sklearn.inspection import permutation_importance

# 1. Calcolo della Permutation Importance
# n_repeats=10 significa che mescola ogni colonna 10 volte per avere una stima robusta
result = permutation_importance(
    estimator=model_pipeline,
    X=X_test,
    y=y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1 # Usa tutti i core del processore per velocizzare
)

# 2. Ordinamento delle feature dalla meno importante alla più importante
sorted_idx = result.importances_mean.argsort()

# 3. Creazione del grafico (Boxplot)
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(
    result.importances[sorted_idx].T,
    vert=False,
    labels=X_test.columns[sorted_idx]
)
ax.set_title("Permutation Feature Importance (Test Set)")
ax.set_xlabel("Accuracy drop (if the feature is shuffled)")
fig.tight_layout()
plt.show()

**1. The X-Axis: The "Performance Drop"**
The horizontal axis measures how much the model worsens (e.g. its Accuracy or F1-Score) when we take a specific column's values and shuffle them randomly, destroying their original relationship with the target.

- **High values (right):** Shuffling that column hurt the model badly → the feature is **crucial**.
- **Values near 0:** The model predicted just as well after shuffling → the feature is **useless or redundant**.
- **Negative values:** The model actually *improved* after shuffling → the feature adds only **noise** and confuses the algorithm (you might even consider removing it).

---

**2. The Y-Axis: The Ranking**
Features are sorted from most important (top) to least important (bottom) — literally what your Random Forest "looks at most" before making a decision.

---

**3. The Boxes / Error Bars**
If you see boxplots or error bars, it's because the shuffle was repeated multiple times (e.g. 10×) to ensure the result isn't random chance.
- **Wide box** → high uncertainty about that feature's importance.
- **Narrow box** → consistent impact across every repetition.

##### 3.5.3.2 Partial Dependence Plots (PDP)
Una volta che il grafico precedente ti ha svelato quali sono le feature più importanti, prendi i nomi delle prime 2 o 3 e inseriscili nei PDP per vedere come influenzano la predizione.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

features_to_plot = ['capital.gain', 'marital.status']

# Creazione della figura
fig, ax = plt.subplots(figsize=(14, 5)) # Ho allargato un po' la figura per il testo

# Generazione del PDP
display = PartialDependenceDisplay.from_estimator(
    estimator=model_pipeline,
    X=X_test,
    features=features_to_plot,
    categorical_features=['marital.status'], # <--- ECCO LA MAGIA: Avvisiamo che è testo!
    kind="average",
    ax=ax
)

display.figure_.suptitle("Partial Dependence Plots")
display.figure_.subplots_adjust(top=0.9)
plt.show()

**1. Capital Gain**
The line is flat for almost all low values, then jumps sharply past the 40k–50k threshold.
This is a **sanity check**: the model correctly learned that only extreme capital gain values push the prediction toward >50K. If the graph had shown something different, the model would be broken.

---

**2. Marital Status**
`Married-civ-spouse` and `Married-AF-spouse` bars are very high, everything else (Never-married, Divorced, Widowed) is near zero.
The model strongly associates being married with a >50K income — likely because married people tend to be older and in a more stable career phase.

#### 3.5.4 Local Interpretability

##### 3.5.4.1 LIME — Local Interpretable Model-agnostic Explanations

In [ ]:
# %whos

In [ ]:
# print(best_model)
# print(model_pipeline)

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
X_train_transformed = best_model.named_steps['preprocessor'].transform(X_train)
X_test_transformed  = best_model.named_steps['preprocessor'].transform(X_test)
feature_names       = best_model.named_steps['preprocessor'].get_feature_names_out()

explainer = LimeTabularExplainer(
    X_train_transformed,
    feature_names=feature_names,
    class_names=['low', 'high'],
    mode='classification'
)

def predict_fn(x):
    return best_model.named_steps['classifier'].predict_proba(x)

exp = explainer.explain_instance(X_test_transformed[0], predict_fn)
exp.as_pyplot_figure()
plt.tight_layout()
plt.show()

To interpret individual predictions, LIME was applied to the first observation of the test set. LIME works by perturbing the input features of a single instance and observing how the model's prediction changes, effectively fitting a simple linear model locally around that point.
The chart shows the contribution of each feature to the prediction of class high (income >50K) for this specific individual:

- num__education.num and cat__sex_Male are the strongest positive drivers, pushing the prediction toward high income
- num__age and cat__education_Bachelors further support the high-income prediction
- cat__marital.status_Never-married is the only strong negative factor, pulling the prediction away from the high-income class — consistent with what was observed in the PDP and error analysis sections, where marital status emerged as one of the most influential features globally

This local explanation is coherent with the global interpretability results: the same features that ranked highest in Permutation Importance and SHAP also dominate the LIME explanation, suggesting the model is internally consistent across both global and local perspectives.Sonnet 4.6

#### 3.5.5 ROC CURVE random forest

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.plot([0, 1], [0, 1], 'k--') # Linea del caso (modello inutile)
plt.title("ROC Curve: Model's discrimination ability")
plt.show()

The model achieves an AUC of 0.91, indicating excellent discriminative power. In practical terms, this means that in 91% of cases the model correctly ranks a high-income individual above a low-income one.
The curve rises steeply toward the top-left corner in the early phase (FPR 0–0.2), meaning the model captures the majority of true high-income cases while keeping false alarms very low. This is the most valuable region of the curve — it shows the model is highly selective when operating at strict thresholds.
The slight flattening after FPR ~0.4 is expected and reflects the inherent difficulty of separating the remaining borderline cases, consistent with the class imbalance observed in the dataset.

#### 3.5.6 Error analysis

In [ ]:
y_pred

In [ ]:
# 2. Merge X_test, true labels (y_test) and predictions into a single DataFrame
# Make sure y_test is an array or series with the same indices as X_test
results_df = X_test.copy()
results_df['True_Value'] = y_test.values if isinstance(y_test, pd.Series) else y_test
results_df['Prediction'] = y_pred

# 3. Isolate the errors
# False Positives: Prediction = 1 (Class 1), but Reality = 0 (Class 0)
false_positives = results_df[(results_df['Prediction'] == 1) & (results_df['True_Value'] == 0)]

# False Negatives: Prediction = 0 (Class 0), but Reality = 1 (Class 1)
false_negatives = results_df[(results_df['Prediction'] == 0) & (results_df['True_Value'] == 1)]

print(f"Total number of False Positives: {len(false_positives)}")
print(f"Total number of False Negatives: {len(false_negatives)}")

# 4. Analyze where the model gets confused (example on a chosen column)
# Replace 'marital.status' with a variable of interest (e.g. 'age' or 'education')
feature_to_analyze = 'marital.status'

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# False Positives chart
sns.countplot(data=false_positives, y=feature_to_analyze, ax=axes[0], palette="Reds_r", order=false_positives[feature_to_analyze].value_counts().index)
axes[0].set_title(f"False Positives by {feature_to_analyze}")
axes[0].set_xlabel("Error Count")

# False Negatives chart
sns.countplot(data=false_negatives, y=feature_to_analyze, ax=axes[1], palette="Blues_r", order=false_negatives[feature_to_analyze].value_counts().index)
axes[1].set_title(f"False Negatives by {feature_to_analyze}")
axes[1].set_xlabel("Error Count")

plt.tight_layout()
plt.show()

1. False Positives — the model is "overconfident"
The red bar for Married-civ-spouse dominates overwhelmingly. This is a direct consequence of what the PDP revealed earlier: the model learned a very strong rule — "if married, likely high income". As a result, it over-applies this rule and misclassifies many married individuals who actually earn ≤50K. The model becomes a victim of its own learned bias.
2. False Negatives — the model is "too conservative"
Married-civ-spouse is still the largest bar simply because it is the most represented group in the dataset. However, notice how Never-married and Divorced are proportionally much more visible here compared to the false positives chart. The model implicitly reasons: "not married → probably low income", causing it to miss many single or divorced individuals who actually earn >50K.

### 3.6 XGBoost
It is an evolution of decision trees. Instead of training 100 independent trees (as in Random Forest), XGBoost trains one tree, evaluates its errors, and then trains the next tree specifically to correct those mistakes.

**Why use it:** It is almost always the top-performing model on tabular data (such as census datasets).

**What to expect:** A ROC curve that is even closer to the top-left corner and an AUC that can potentially reach 0.92–0.94.


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# Creiamo la pipeline con XGBoost
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric='logloss', random_state=42))
])

# Definiamo i parametri per il Tuning (fondamentale per XGBoost)
param_grid_xgb = {
    'classifier__n_estimators': [100],
    'classifier__learning_rate': [0.1], # La velocità di correzione degli errori
    'classifier__max_depth': [3, 6],      # Profondità degli alberi
    'classifier__subsample': [0.8, 1.0]       # % di dati usati per ogni albero
}

grid_xgb = GridSearchCV(xgb_pipeline, param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print(f"Best AUC with XGBoost: {grid_xgb.best_score_:.4f}")

In [ ]:
grid_xgb.best_params_

#### 3.6.1 Log loss xgboost

In [ ]:
# 1. Estraiamo i migliori parametri e il preprocessor
best_params = grid_xgb.best_params_
# Rimuoviamo il prefisso 'classifier__' dai nomi dei parametri per passatli a XGB
xgb_params = {k.replace('classifier__', ''): v for k, v in best_params.items()}

# 2. Trasformiamo i dati una volta sola per velocizzare
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# 3. Alleniamo il modello finale monitorando l'errore
final_xgb = XGBClassifier(**xgb_params, eval_metric='logloss', random_state=42)
final_xgb.fit(X_train_proc, y_train,
              eval_set=[(X_train_proc, y_train), (X_test_proc, y_test)],
              verbose=False)

# 4. Grafico della Log Loss
results = final_xgb.evals_result()
plt.figure(figsize=(10, 5))
plt.plot(results['validation_0']['logloss'], label='Train Loss')
plt.plot(results['validation_1']['logloss'], label='Test Loss')
plt.title('Log Loss: Monitoraggio dell\'apprendimento e Overfitting')
plt.xlabel('Numero di Alberi')
plt.ylabel('Log Loss')
plt.legend()
plt.show()

Both curves drop rapidly at the beginning and then start to flatten out. This is normal and positive—the model learns quickly in the early trees, and improvements become marginal afterward.

**The critical point: the gap between the two curves**

* Train Loss (blue) → 0.256 (continues decreasing)
* Test Loss (orange) → 0.288 (plateaus earlier)

There is a growing gap between train and test, especially after around 40 trees. This is a sign of mild overfitting—the model starts to memorize the training set instead of generalizing.

**Is it concerning?**

* Small gap (~0.03): Mild, not severe
* Test loss stabilizes: Good sign
* Train loss keeps decreasing: Confirms slight overfitting

In this case, it’s not alarming, but it suggests that 100 trees might be too many. The test loss stops improving around 60–70 trees.

**What to do**

You can add early stopping to automatically stop training when the test loss no longer improves.


#### 3.6.2 SHAP xgboost

In [ ]:
import shap

# 1. estrai il modello dalla pipeline
xgb_model = grid_xgb.best_estimator_.named_steps['classifier']

# 2. trasforma X_test con il preprocessor
X_test_transformed = grid_xgb.best_estimator_.\
                     named_steps['preprocessor']\
                     .transform(X_test)

# 3. crea l'explainer
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_transformed)

##### 3.6.2.1 Chart 1 — Global Importance (bar chart)
Answers the question: *“Which features matter the most on average?”*

* Feature 29 → 1.1   ← the most important overall
* Feature 0  → 0.68
* Feature 2  → 0.47
* ...
* Feature 45 → ~0.02  ← almost irrelevant

This represents the mean absolute SHAP value across the entire test set. It shows the magnitude of importance, but not the direction of the effect.


In [ ]:
shap.summary_plot(shap_values, X_test_transformed, plot_type="bar")

In [ ]:
# se usi get_feature_names_out dal preprocessor
feature_names = grid_xgb.best_estimator_\
                .named_steps['preprocessor']\
                .get_feature_names_out()

# print(feature_names[29])   # ti dice il nome reale
# print(list(enumerate(feature_names)))  # lista completa indice → nome

##### 3.6.2.2 Graph 2 — Beeswarm plot

It answers: **“In which direction and with what intensity does each feature influence the prediction?”**

Each point represents **a single row in your dataset**. The color should indicate red/blue (high/low feature value), but in your plot it appears gray — likely due to a rendering issue.

What you can observe:

* **Feature 2** has points reaching up to +8 → extreme cases that strongly push the prediction toward class 1
* **Feature 29** has points spread on both the left and right → it can push predictions in both directions
* **Feature 3** shows a very wide distribution → highly variable behavior

Feature mapping:

* Feature 2: `num__capital.gain` (Capital gains)
* Feature 3: `num__capital.loss` (Capital losses)
* Feature 29: `cat__marital.status_Married-civ-spouse` (Marital status: Married-civ-spouse)


In [ ]:
shap.summary_plot(shap_values, X_test_transformed)

##### 3.6.2.3 Graph 3 — Waterfall plot (single prediction)

It answers: **“Why did the model produce this result for this specific individual?”**

* **E[f(X)] = -1.188** ← starting point (average model output across all data)
* **f(x) = -1.979** ← final prediction for this individual

Effectively, the plot shows how each feature contribution moves the prediction from the baseline (average) to the final value for this specific case.


In [ ]:
shap.waterfall_plot(explainer(X_test_transformed)[0])

<!-- | Feature | Valore | SHAP | Effetto |
| :--- | :--- | :--- | :--- |
| **Feature 2** | 90 | -1.33 | spinge forte verso classe 0 |
| **Feature 1** | 1 | +0.76 | spinge verso classe 1 |
| **Feature 0** | -0.474 | -0.56 | spinge verso classe 0 |
| **Feature 4** | 1 | +0.39 | spinge verso classe 1 |

La predizione finale -1.979 è il risultato di tutte queste spinte sommate al punto di partenza. Essendo negativa, il modello classifica questa persona come classe 0. -->

### 3.7 Cluster

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [ ]:
import matplotlib.pyplot as plt

# 1. Usiamo il preprocessor per avere i dati pronti (scalati e codificati)
X_preprocessed = preprocessor.fit_transform(X)

inerzia = []
K_range = range(1, 11) # Proviamo da 1 a 10 cluster

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_preprocessed)
    inerzia.append(km.inertia_)

# 2. Plot del grafico
plt.figure(figsize=(10, 6))
plt.plot(K_range, inerzia, marker='o', linestyle='--')
plt.xlabel('Cluster Number (K)')
plt.ylabel('Inertia')
plt.title('Elbow Test to identify the optimal number of Clusters')
plt.xticks(K_range)
plt.grid(True)
plt.show()

In [ ]:
# Eseguiamo il K-Means finale con K scelto (es. 2)
kmeans_final = KMeans(n_clusters=3, random_state=42)
clusters = kmeans_final.fit_predict(X_preprocessed)

# Aggiungiamo i cluster al DataFrame originale per confrontare
df_1['cluster'] = clusters

# Vediamo come si distribuisce il reddito nei cluster
print(pd.crosstab(df_1['cluster'], df_1['is_high_income']))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Creiamo la tabella percentuale
ct_pct = pd.crosstab(df_1['cluster'], df_1['is_high_income'], normalize='index')

# Grafico
ct_pct.plot(kind='bar', stacked=True, figsize=(8, 5), color=['#ff9999', '#66b3ff'])
plt.title("Income Composition by Cluster (Percentage)")
plt.ylabel("Proportion")
plt.legend(title="Income >50K", labels=["No", "Yes"])
plt.show()

### 3.8 Deep Learning

In [ ]:
from sklearn.neural_network import MLPClassifier

# Definiamo la Pipeline per il Deep Learning
mlp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        activation='relu',
        solver='adam',
        random_state=42,
        early_stopping=True # Evita l'overfitting fermandosi quando l'errore non scende più
    ))
])

# Training
mlp_pipeline.fit(X_train, y_train)

## 4. ROC curve final comparison

In [ ]:
from sklearn.metrics import roc_curve, auc

# Calcoliamo le probabilità per tutti e tre
y_prob_rf = best_model.predict_proba(X_test)[:, 1]
y_prob_xgb = grid_xgb.predict_proba(X_test)[:, 1]
y_prob_mlp = mlp_pipeline.predict_proba(X_test)[:, 1]

# Funzione per plottare tutto insieme
plt.figure(figsize=(10, 8))
for name, prob in [("Random Forest", y_prob_rf), ("XGBoost", y_prob_xgb), ("Deep Learning", y_prob_mlp)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.title("Final comparison: RF vs XGBoost vs MLP")
plt.legend()
plt.show()